# Bản Đồ Nhiệt (Choropleth Map) - Phân Bố Thị Trường Xe Cũ 63 Tỉnh/Thành

Notebook này trực quan hóa phân bổ lượng tin đăng và giá bán trung bình trên 63 tỉnh/thành phố tại Việt Nam:
1. **Thang đo logarit ($log_{10}$):** Xử lý độ lệch phân bố cực lớn giữa hai trung tâm (Hà Nội, TP.HCM) và các địa phương khác, giúp dải màu thể hiện rõ nét sự phân hóa.
2. **Đánh dấu độ tin cậy mẫu:** 14 tỉnh thành có dưới 10 tin đăng được gắn nhãn cảnh báo độ tin cậy thấp kèm điểm đánh dấu trực quan trên bản đồ.
3. **Tương tác trực quan (Hover):** Hiển thị chi tiết tên tỉnh, số tin đăng, giá trung bình và trạng thái tin cậy khi rê chuột.
4. **Dữ liệu GeoJSON nội bộ:** Đọc trực tiếp từ thư mục `data/geojson/` trong repository, không phụ thuộc kết nối internet bên ngoài.

In [1]:
from pathlib import Path
import json
import warnings
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

warnings.filterwarnings('ignore')

# Cấu hình renderer chuyên dụng cho VS Code và Jupyter
pio.renderers.default = 'plotly_mimetype'

# Thiết lập đường dẫn động từ thư mục gốc của repository
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())
DATA_PATH = ROOT / "data" / "processed" / "cars_cleaned.csv"
GEOJSON_PATH = ROOT / "data" / "geojson" / "vietnam_provinces.geojson"
CENTROIDS_PATH = ROOT / "data" / "geojson" / "province_centroids.json"

print(f"Nạp dữ liệu từ: {DATA_PATH.name}")
print(f"Nạp GeoJSON từ: {GEOJSON_PATH.name}")

Nạp dữ liệu từ: cars_cleaned.csv
Nạp GeoJSON từ: vietnam_provinces.geojson


### 1. Đọc và Tiền Xử Lý Dữ Liệu
Tính toán tổng số lượng tin đăng và giá bán trung bình theo từng tỉnh/thành phố từ bộ dữ liệu sạch `cars_cleaned.csv`.

In [2]:
df = pd.read_csv(DATA_PATH, low_memory=False)

province_stats = df.groupby('region_name').agg(
    so_tin=('list_id', 'count'),
    gia_trung_binh=('price_million', 'mean')
).reset_index()

province_stats['gia_trung_binh'] = province_stats['gia_trung_binh'].round(1)

print(f"Tổng số tỉnh/thành có tin đăng: {len(province_stats)}/63")
province_stats.sort_values(by='so_tin', ascending=False).head(10)

Tổng số tỉnh/thành có tin đăng: 63/63


,region_name,so_tin,gia_trung_binh
24,Hồ Chí Minh,6149,514.3
17,Hà Nội,2467,475.6
2,Bình Dương,873,399.8
13,Cần Thơ,529,396.4
61,Đồng Nai,510,368.3
58,Đà Nẵng,509,391.8
59,Đắk Lắk,376,352.7
31,Lâm Đồng,164,415.8
0,An Giang,156,400.3
1,Bà Rịa - Vũng Tàu,146,366.5


### 2. Đánh Giá Độ Tin Cậy & Áp Dụng Thang Logarit
- **Thang logarit:** Hai đô thị lớn (TP.HCM và Hà Nội) chiếm phần lớn thị phần. Nếu sử dụng thang tuyến tính, toàn bộ các tỉnh thành còn lại sẽ bị hiển thị mờ nhạt do giá trị quá nhỏ so với cực đại. Thang $log_{10}$ giúp trải đều dải màu cho các tỉnh từ vài chục đến hàng trăm tin.
- **Độ tin cậy:** Các tỉnh có dưới 10 tin đăng bị xem là "Không đủ tin cậy (< 10 tin)" do cỡ mẫu quá nhỏ, giá trung bình có thể bị sai lệch bởi một vài xe cá biệt.

In [3]:
province_stats['do_tin_cay'] = province_stats['so_tin'].apply(
    lambda x: 'Đáng tin cậy' if x >= 10 else 'Không đủ tin cậy (< 10 tin)'
)

# Thang log10 (cộng 1 để tránh lỗi log(0))
province_stats['log_so_tin'] = np.log10(province_stats['so_tin'] + 1)

# Thống kê phân loại độ tin cậy
print(province_stats['do_tin_cay'].value_counts())
print("\nDanh sách 14 tỉnh dưới 10 tin đăng:")
print(province_stats[province_stats['so_tin'] < 10][['region_name', 'so_tin', 'gia_trung_binh']].to_string(index=False))

do_tin_cay
Đáng tin cậy                   49
Không đủ tin cậy (< 10 tin)    14
Name: count, dtype: int64

Danh sách 14 tỉnh dưới 10 tin đăng:
region_name  so_tin  gia_trung_binh
   Bạc Liêu       5           273.0
    Bắc Kạn       1           405.0
   Cao Bằng       2           446.5
     Cà Mau       8           233.1
   Hà Giang       3           575.0
   Lai Châu       1           130.0
    Lào Cai       3           388.3
   Lạng Sơn       5           559.6
 Ninh Thuận       8           364.1
 Quảng Bình       5           334.0
     Sơn La       4           372.2
Tuyên Quang       8           324.6
    Yên Bái       4           265.0
  Điện Biên       1           610.0


### 3. Nạp Dữ Liệu Địa Lý GeoJSON Nội Bộ
Đọc ranh giới 63 tỉnh/thành và tọa độ tâm (centroid) từ thư mục `data/geojson/`.

In [4]:
with open(GEOJSON_PATH, 'r', encoding='utf-8') as f:
    vn_geojson = json.load(f)

with open(CENTROIDS_PATH, 'r', encoding='utf-8') as f:
    province_centroids = json.load(f)

geo_provinces = {feat['properties']['Ten_Clean'] for feat in vn_geojson['features']}
data_provinces = set(province_stats['region_name'])

matched = data_provinces.intersection(geo_provinces)
print(f"Độ khớp tên tỉnh giữa dữ liệu và GeoJSON: {len(matched)}/63 tỉnh")

Độ khớp tên tỉnh giữa dữ liệu và GeoJSON: 63/63 tỉnh


### 4. Bản Đồ Nhiệt 63 Tỉnh/Thành (Choropleth Map)
Cấu trúc bản đồ gồm:
- **Lớp Choropleth:** Thể hiện lượng tin đăng theo dải màu thang logarit `YlOrRd`. Thanh chú thích (Colorbar) hiển thị các mốc số lượng tin thực tế (1, 10, 100, 1.000, 10.000 tin).
- **Lớp Cảnh Báo Trực Quan:** Điểm tam giác đỏ đặt tại 14 tỉnh có dưới 10 tin giúp người xem nhận biết ngay vùng trũng dữ liệu.
- **Hover Chi Tiết:** Rê chuột vào từng tỉnh hiển thị tên tỉnh, số tin đăng, giá trung bình và mức độ tin cậy.

In [5]:
# 1. Vẽ bản đồ nhiệt theo thang logarit
fig = px.choropleth(
    province_stats,
    geojson=vn_geojson,
    locations='region_name',
    featureidkey='properties.Ten_Clean',
    color='log_so_tin',
    color_continuous_scale='YlOrRd',
    hover_name='region_name',
    hover_data={
        'so_tin': True,
        'gia_trung_binh': True,
        'do_tin_cay': True,
        'log_so_tin': False
    },
    labels={
        'so_tin': 'Số tin đăng',
        'gia_trung_binh': 'Giá TB (Triệu VNĐ)',
        'do_tin_cay': 'Mức độ tin cậy'
    },
    title='Bản Đồ Phân Bố Lượng Xe Cũ & Giá Trung Bình 63 Tỉnh/Thành'
)

# 2. Đánh dấu các tỉnh có dưới 10 tin đăng
low = province_stats[province_stats['so_tin'] < 10]
fig.add_trace(go.Scattergeo(
    lon=low['region_name'].map(lambda x: province_centroids.get(x, {}).get('lon')),
    lat=low['region_name'].map(lambda x: province_centroids.get(x, {}).get('lat')),
    mode='markers',
    marker=dict(size=8, color='red', symbol='triangle-up'),
    name='Cảnh báo: Dưới 10 tin (Mẫu nhỏ)'
))

# 3. Tinh chỉnh hiển thị bản đồ
fig.update_geos(fitbounds='locations', visible=False)
fig.update_coloraxes(
    colorbar_title='Số tin',
    colorbar_tickvals=[1, 2, 3, 4],
    colorbar_ticktext=['10', '100', '1.000', '10.000']
)
fig.update_layout(height=650, margin=dict(r=0, t=50, l=0, b=0))

fig.show()

### 5. Kết Luận Phân Tích Địa Lý Thị Trường
1. **Thị trường tập trung cao độ tại hai đầu cực:** TP.HCM và Hà Nội là hai thị trường giao dịch xe cũ sôi động nhất cả nước, chiếm hơn **65%** tổng lượng tin rao bán trên toàn quốc.
2. **Phân vùng thanh khoản thứ cấp:** Các địa phương công nghiệp và vệ tinh lớn (Bình Dương, Đồng Nai, Đà Nẵng, Hải Phòng) có lượng tin giao dịch ổn định từ 200 - 600 tin.
3. **Lưu ý định giá tại các tỉnh mẫu nhỏ:** 14 tỉnh vùng núi và vùng xa (được đánh dấu tam giác đỏ) có dưới 10 tin đăng. Mức giá trung bình tại các khu vực này có độ biến động lớn và chưa đủ độ tin cậy thống kê để làm cơ sở định giá chung.